# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and prepare for analysis the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. All dataset structures (record sets, fields, etc.) are referenced by their Croissant `@id` for reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and fields. In Croissant, every entity is referenced by its unique `@id`.

We'll print out the available record sets and fields in this dataset.

In [ ]:
# List all record sets defined in the dataset metadata
print("Available Record Sets (by @id):")
record_sets = [rs for rs in metadata.record_sets]
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")

# For each record set, print its fields and columns
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - {field['@id']}", end='')
            if 'name' in field:
                print(f" (name: {field['name']})", end='')
            print()
    if 'column' in rs:
        print("  Columns:")
        for col in rs['column']:
            print(f"    - {col['@id']}", end='')
            if 'name' in col:
                print(f" (name: {col['name']})", end='')
            print()


## 3. Data Extraction
Load data from a record set into a DataFrame for exploration.

**Note:** We reference record sets, fields, and columns by their unique `@id`.

See the cell above for available record set `@id`s. We'll use the main survey table (replace below with the correct `@id` for the main table in the dataset).

In [ ]:
# Example: Extract data from the primary survey record set
# Replace with the correct @id of the main record set from the overview! See previous cell.
main_record_set_id = 'cr:RecordSet'  # (Example, adjust as needed from printed record sets)

dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records for record set {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Preview:\n{df.head()}\n")

# For convenience, pick one record set to use going forward
df = dataframes[main_record_set_id]
print(f"Using record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data wrangling steps to the DataFrame: filtering, normalization, grouping, etc.

We use field or column `@id` for selection. See the columns printed in the previous cell.


In [ ]:
# Please adjust these field/column @id values based on the outputs printed above
# For demonstration, we'll assume there's a column '@id': 'cr:field_psq_total_score' for PSQ total score (numeric)

numeric_field = 'cr:field_psq_total_score'  # Replace with appropriate @id
group_field = 'cr:field_gender'  # Replace with appropriate @id (e.g., for gender)

if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold}: {len(filtered_df)} records")
    
    # Normalize
    filtered_df[numeric_field + "_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

    # Group by another field if present
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} by {group_field}:")
        print(grouped_df)
else:
    print(f"Numeric field {numeric_field} not found in DataFrame columns. Update the field to match a numeric column @id from the overview.")

## 5. Visualization
Visualize the distribution of a numeric field and grouped means (if grouping was possible).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # Grouped boxplot if group field available
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library. We examined record set and field structures via Croissant `@id`, extracted a survey table, filtered and normalized data for a numeric field, grouped by a demographic attribute, and visualized data distributions. For further analysis, adjust field and record set `@id`s as appropriate for your questions.